## Salesforce Codegen 350M
We will start with the 350M pre-trained Salesforce codegen-350M model and work in improving
its inference speed using the techniques from the last 2 chapters.

In [68]:
!pip install optimum[onnxruntime]

In [69]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [70]:
import torch
from transformers import AutoTokenizer, CodeGenForCausalLM

device = "cpu"
model_id = "Salesforce/codegen-350M-mono"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = CodeGenForCausalLM.from_pretrained(model_id).to(device)
model.eval()
model

Some weights of the model checkpoint at Salesforce/codegen-350M-mono were not used when initializing CodeGenForCausalLM: ['transformer.h.0.attn.causal_mask', 'transformer.h.1.attn.causal_mask', 'transformer.h.10.attn.causal_mask', 'transformer.h.11.attn.causal_mask', 'transformer.h.12.attn.causal_mask', 'transformer.h.13.attn.causal_mask', 'transformer.h.14.attn.causal_mask', 'transformer.h.15.attn.causal_mask', 'transformer.h.16.attn.causal_mask', 'transformer.h.17.attn.causal_mask', 'transformer.h.18.attn.causal_mask', 'transformer.h.19.attn.causal_mask', 'transformer.h.2.attn.causal_mask', 'transformer.h.3.attn.causal_mask', 'transformer.h.4.attn.causal_mask', 'transformer.h.5.attn.causal_mask', 'transformer.h.6.attn.causal_mask', 'transformer.h.7.attn.causal_mask', 'transformer.h.8.attn.causal_mask', 'transformer.h.9.attn.causal_mask']
- This IS expected if you are initializing CodeGenForCausalLM from the checkpoint of a model trained on another task or with another architecture (e

CodeGenForCausalLM(
  (transformer): CodeGenModel(
    (wte): Embedding(51200, 1024)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-19): 20 x CodeGenBlock(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): CodeGenAttention(
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
          (qkv_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (mlp): CodeGenMLP(
          (fc_in): Linear(in_features=1024, out_features=4096, bias=True)
          (fc_out): Linear(in_features=4096, out_features=1024, bias=True)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=51200, bias=True)
)

The CodeGenForCausalLM class is a specialized model head within the Hugging Face transformers library, specifically designed for the Salesforce CodeGen family of models. It is built to handle Causal Language Modeling, which is the technical way of saying "predicting the next token in a sequence" (autoregression).

While it shares many similarities with standard models like GPT2LMHeadModel, it has specific architectural nuances tailored for code generation.

Core Functions of the Class
The primary job of CodeGenForCausalLM is to take the hidden states from the Transformer's "body" and map them back into the vocabulary space so the model can choose a word or code snippet.

Logit Computation: It applies a linear layer (the "LM Head") to the final hidden states to transform a vector of size hidden_size into a vector the size of the vocab_size.

Loss Calculation: If you provide "labels" during the forward pass, the class automatically calculates the Cross-Entropy Loss. This makes it ready for fine-tuning out of the box.

**Parallel Architecture Support**: Unlike standard GPT-2, CodeGen uses a parallelized Transformer block (similar to GPT-J). This class is specifically wired to handle that parallel flow where the MLP and Attention layers are computed at the same time rather than sequentially.

In [71]:
prompt = "def hello_world():"
inputs = tokenizer(prompt, return_tensors="pt")

gen = model.generate(**inputs)
output = tokenizer.decode(gen[0],
                       skip_special_tokens=True,
                       pad_token_id=50256)
print(output)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


def hello_world():
    print("Hello World")

hello_world()

# 파


In [72]:
# Let's write a method for the token generation and decoding so that we can reuse it with different prompts
def generate_text(prompt, tokenizer, model):
    inputs = tokenizer(prompt, return_tensors="pt")
    gen = model.generate(**inputs)
    output = tokenizer.decode(gen[0],
                              skip_special_tokens=True,
                              pad_token=tokenizer.eos_token_id)
    return output


In [73]:
# Let's write a method for the token generation and decoding so that we can reuse it with different prompts
# This version only passes the input_ids to the model and not the attention mask
def generate_text_from_book(prompt, tokenizer, model):
    inputs_ids = tokenizer(prompt, return_tensors="pt").input_ids
    gen = model.generate(inputs_ids, max_length=12)
    output = tokenizer.decode(gen[0], skip_special_tokens=True, pad_token=tokenizer.eos_token_id)
    return output

In [74]:
prompt = "Create bar chart with matplotlib"
output = generate_text(prompt, tokenizer, model)
print(output)

output = generate_text_from_book(prompt, tokenizer, model)
print(output)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Create bar chart with matplotlib

# Create a bar chart with the data

# Create a bar chart with the data
Create bar chart with matplotlib

# Create a


In [75]:
from optimum.onnxruntime import ORTModelForCausalLM
model_id = 'Salesforce/codegen-350M-mono'
model_onnx = ORTModelForCausalLM.from_pretrained(model_id,
    export=True,
    provider="CPUExecutionProvider"
)

Some weights of the model checkpoint at Salesforce/codegen-350M-mono were not used when initializing CodeGenForCausalLM: ['transformer.h.0.attn.causal_mask', 'transformer.h.1.attn.causal_mask', 'transformer.h.10.attn.causal_mask', 'transformer.h.11.attn.causal_mask', 'transformer.h.12.attn.causal_mask', 'transformer.h.13.attn.causal_mask', 'transformer.h.14.attn.causal_mask', 'transformer.h.15.attn.causal_mask', 'transformer.h.16.attn.causal_mask', 'transformer.h.17.attn.causal_mask', 'transformer.h.18.attn.causal_mask', 'transformer.h.19.attn.causal_mask', 'transformer.h.2.attn.causal_mask', 'transformer.h.3.attn.causal_mask', 'transformer.h.4.attn.causal_mask', 'transformer.h.5.attn.causal_mask', 'transformer.h.6.attn.causal_mask', 'transformer.h.7.attn.causal_mask', 'transformer.h.8.attn.causal_mask', 'transformer.h.9.attn.causal_mask']
- This IS expected if you are initializing CodeGenForCausalLM from the checkpoint of a model trained on another task or with another architecture (e

In [76]:
onnx_path = Path("/content/drive/MyDrive/onnx_models/codegen-350m-onnx")
model_onnx.save_pretrained(onnx_path)

In [78]:
from optimum.onnxruntime import ORTQuantizer
from optimum.onnxruntime.configuration import AutoQuantizationConfig

dynamic_quantizer = ORTQuantizer.from_pretrained(model_onnx)
dqconfig = AutoQuantizationConfig.avx512_vnni(is_static=False,
                                              per_channel=False)

model_quantized_path = dynamic_quantizer.quantize(save_dir=onnx_path,
                                                  quantization_config=dqconfig
                                                  )

Step-by-Step Breakdown
1. ORTQuantizer.from_pretrained(model)
This initializes the quantization engine. It loads your existing ONNX model and prepares it for the transformation. It inspects the graph to identify which operations (like MatMul or Gemm) can be safely converted to integer math.

2. AutoQuantizationConfig.avx512_vnni(...)
This is a high-level "preset" configuration designed specifically for Intel CPUs (specifically those with AVX-512 and VNNI instruction sets).
- AVX-512 / VNNI: These are hardware-level accelerators in the CPU that can multiply 8-bit integers in massive batches.
- is_static=False (Dynamic): This tells the model to calculate the scaling factors for the activations on the fly during inference. It’s the "easy mode" of quantization because you don't need a calibration dataset.
- per_channel=False: This means the model uses one scaling factor for the entire weight tensor rather than a separate one for each output channel. It’s slightly less accurate but faster on some older CPU architectures.

3. dynamic_quantizer.quantize(...)
This is the execution phase. It processes the model graph, applies the config, and saves the new, shrunken .onnx file to your onnx_path.

In [79]:
print(model_quantized_path)

/content/drive/MyDrive/onnx_models/codegen-350m-onnx


In [80]:
import os

original_model_path = os.path.join(onnx_path, "model.onnx")
quantized_model_path = os.path.join(onnx_path, "model_quantized.onnx")

original_size = os.path.getsize(original_model_path) / (1024 * 1024) # Size in MB
quantized_size = os.path.getsize(quantized_model_path) / (1024 * 1024) # Size in MB

print(f"Original model size: {original_size:.2f} MB")
print(f"Quantized model size: {quantized_size:.2f} MB")
print(f"Size reduction: {((original_size - quantized_size) / original_size) * 100:.2f}%")

Original model size: 1361.74 MB
Quantized model size: 342.19 MB
Size reduction: 74.87%


In [81]:
# We will use the Hugging Face pipeline to define the inference that needs to be
# done so that we can apply it to the three models to assess the performance
# improvement
#
from transformers import pipeline, GenerationConfig
# The GenerationConfig object is no longer needed here, as its parameters will be passed directly
# generation_config = GenerationConfig(pad_token_id=50256,
#                                      truncation=True,
#                                      max_length=12)
pipe = pipeline("text-generation",
                model=model,
                tokenizer=tokenizer,
                device="cpu", # Explicitly set device to CPU
                pad_token_id=50256, # Pass pad_token_id directly
                truncation=True,    # Pass truncation directly
                max_length=12       # Pass max_length directly
)

Device set to use cpu


In [82]:
# Test the pipeline
#
result = pipe(prompt)
print(result[0]['generated_text'])

Create bar chart with matplotlib's pyplot interface
        barchart = py.figure(figsize=(6, 3))
        barcode_data = py.figure()
        py.plot(values, linew


In [83]:
# Define some methods for collecting metrics during inference
#
from contextlib import contextmanager
from time import perf_counter
from dataclasses import dataclass

@contextmanager
def track_infer_time(time_buffer):
  start_time = perf_counter()
  yield
  end_time = perf_counter()
  time_buffer.append(end_time - start_time)

@dataclass
class BenchmarkInferenceResult:
  model_inference_time: [int]
  optimized_model_path: str


In [84]:
from tqdm import trange
PROVIDERS = {
    ("cpu", "PyTorch CPU"),
}
results = {}
inference_runs = 200
for device, label in PROVIDERS:
  time_buffer = []

  for _ in trange(inference_runs, desc=f"Tracking inference time (PyTorch vanilla model)"):
    with track_infer_time(time_buffer):
      pipe(prompt)

  results[label] = BenchmarkInferenceResult(time_buffer, None)

Tracking inference time (PyTorch vanilla model): 100%|██████████| 200/200 [07:06<00:00,  2.13s/it]


In [85]:
# Delete the model and clear the memory before repeating the performance evaluation
# of the onnx model.
#
import gc

del pipe
del model
gc.collect()

4305

In [86]:
# Load the ONNX version of the model from the onnx_path
#
from optimum.onnxruntime import ORTModelForCausalLM

model_onnx = ORTModelForCausalLM.from_pretrained(onnx_path, provider="CPUExecutionProvider", file_name="model.onnx")

In [87]:
# Define the pipeline and test to make sure it works
#
pipe = pipeline("text-generation",
                model=model_onnx,
                device="cpu", # Explicitly set device to CPU
                tokenizer=tokenizer,
                pad_token_id=50256, # Pass pad_token_id directly
                truncation=True,    # Pass truncation directly
                max_length=12       # Pass max_length directly
)

result = pipe(prompt)
print(result[0]['generated_text'])

Device set to use cpu


Create bar chart with matplotlib module

# Plotting Data
fig = plt.figure( figsize=( 6 * 2, 3.5 * 2) )
#fig = plt.figure(figsize=(10,


In [88]:
from tqdm import trange

PROVIDERS = {
    ("CPUExecutionProvider", "ONNX CPU"),
}
inference_runs = 200
for device, label in PROVIDERS:
  time_buffer = []

  for _ in trange(inference_runs, desc=f"Tracking inference time (ONNX model)"):
    with track_infer_time(time_buffer):
      pipe(prompt)

  results[label] = BenchmarkInferenceResult(time_buffer, None)

Tracking inference time (ONNX model): 100%|██████████| 200/200 [05:05<00:00,  1.53s/it]


In [89]:
print(results)

{'PyTorch CPU': BenchmarkInferenceResult(model_inference_time=[2.1825043910002933, 2.371705896000094, 2.1657266280008116, 2.1117550439994375, 2.092701202000171, 2.106975591000264, 2.1127226529997643, 2.170117577000383, 2.406468211000174, 2.1729288730002736, 2.110897699000816, 2.1094027829994957, 2.1193907719998606, 2.1611978880000606, 2.7719895640002505, 2.101817363000009, 2.10524485499991, 1.67466616099955, 2.096990961999836, 2.1244238359995506, 2.382627081999999, 1.2592235519996393, 2.134293083000557, 2.1138535480004066, 2.1056617670001287, 2.1028213330000654, 2.1344352309997703, 2.236671712000316, 2.2040483419996235, 2.0881821969996963, 2.0762526429998616, 2.083045039999888, 2.096356048000416, 2.148231128000589, 2.1721249590000298, 2.1500385129993447, 2.112044593999599, 2.4086751449995063, 2.111207397000726, 2.136773925999478, 2.7416910279998774, 2.477042339000036, 2.3346665939998275, 2.0951835220002977, 2.1034464989998014, 2.133972537999398, 2.1719829220000975, 2.24902385400037, 2.

In [90]:
del pipe
del model_onnx
gc.collect()

0

In [91]:
# Load the quantized version of the model from
print(f'Loading the model from: {model_quantized_path}')
model_quantized = ORTModelForCausalLM.from_pretrained(onnx_path, provider="CPUExecutionProvider", file_name="model_quantized.onnx")


Loading the model from: /content/drive/MyDrive/onnx_models/codegen-350m-onnx


In [92]:
# Define the pipeline and test to make sure it works
#
pipe = pipeline("text-generation",
                model=model_quantized,
                device="cpu", # Explicitly set device to CPU
                tokenizer=tokenizer,
                pad_token_id=50256, # Pass pad_token_id directly
                truncation=True,    # Pass truncation directly
                max_length=12       # Pass max_length directly
)

result = pipe(prompt)
print(result[0]['generated_text'])

Device set to use cpu


Create bar chart with matplotlib bar chart from the previous data (i.e. Bar Chart with title "Veg Veg V")
    bar_chart = (fetch_data_BarChart(series_name, '2015


In [93]:
PROVIDERS = {
    ("CPUExecutionProvider", "Quantized ONNX CPU"),
}
inference_runs = 200
for device, label in PROVIDERS:
  time_buffer = []

  for _ in trange(inference_runs, desc=f"Tracking inference time {label}"):
    with track_infer_time(time_buffer):
      pipe(prompt)

  results[label] = BenchmarkInferenceResult(time_buffer, None)

Tracking inference time Quantized ONNX CPU: 100%|██████████| 200/200 [03:16<00:00,  1.02it/s]


In [94]:
import numpy as np
import plotly.express as px

def plot_results(results):
  # Compute average inference time
  time_results = {k: np.mean(v.model_inference_time) * 1e3 for k, v in results.items()}

  fig = px.bar(x=time_results.keys(), y=time_results.values(),
              title="Average inference time (ms) for each provider",
              labels={'x':'Provider', 'y':'Avg Inference time (ms)'},
              text_auto='.2s')
  fig.show()

In [95]:
plot_results(results)

In [96]:
# Save the results dictionary into a JSON file so that it can be reloaded later without having to rerun all the inference benchmarks
#
import json
from pathlib import Path

results_path = Path("/content/drive/MyDrive/onnx_models/results.json")

# Prepare the results for JSON serialization
serializable_results = {}
for key, benchmark_result in results.items():
    serializable_results[key] = {
        "model_inference_time": benchmark_result.model_inference_time,
        "optimized_model_path": str(benchmark_result.optimized_model_path) if benchmark_result.optimized_model_path else None
    }

with open(results_path, "w") as f:
    json.dump(serializable_results, f, indent=4)

print(f"Results saved to {results_path}")

Results saved to /content/drive/MyDrive/onnx_models/results.json


In [97]:
del pipe
del model_quantized
gc.collect()

979

In [98]:
# load the results from the json file
#
import json
from pathlib import Path

results_path = Path("/content/drive/MyDrive/onnx_models/results.json")

# Define the BenchmarkInferenceResult dataclass again, as it's needed for reconstruction
from dataclasses import dataclass

@dataclass
class BenchmarkInferenceResult:
  model_inference_time: list
  optimized_model_path: str

loaded_serializable_results = {}
with open(results_path, "r") as f:
    loaded_serializable_results = json.load(f)

# Reconstruct the results dictionary with BenchmarkInferenceResult objects
loaded_results = {}
for key, data in loaded_serializable_results.items():
    loaded_results[key] = BenchmarkInferenceResult(
        model_inference_time=data["model_inference_time"],
        optimized_model_path=data["optimized_model_path"]
    )

print("Results loaded successfully:")
print(loaded_results)

Results loaded successfully:
{'PyTorch CPU': BenchmarkInferenceResult(model_inference_time=[2.1825043910002933, 2.371705896000094, 2.1657266280008116, 2.1117550439994375, 2.092701202000171, 2.106975591000264, 2.1127226529997643, 2.170117577000383, 2.406468211000174, 2.1729288730002736, 2.110897699000816, 2.1094027829994957, 2.1193907719998606, 2.1611978880000606, 2.7719895640002505, 2.101817363000009, 2.10524485499991, 1.67466616099955, 2.096990961999836, 2.1244238359995506, 2.382627081999999, 1.2592235519996393, 2.134293083000557, 2.1138535480004066, 2.1056617670001287, 2.1028213330000654, 2.1344352309997703, 2.236671712000316, 2.2040483419996235, 2.0881821969996963, 2.0762526429998616, 2.083045039999888, 2.096356048000416, 2.148231128000589, 2.1721249590000298, 2.1500385129993447, 2.112044593999599, 2.4086751449995063, 2.111207397000726, 2.136773925999478, 2.7416910279998774, 2.477042339000036, 2.3346665939998275, 2.0951835220002977, 2.1034464989998014, 2.133972537999398, 2.171982922

In [99]:
plot_results(loaded_results)

In [100]:
time_results = {k: np.mean(v.model_inference_time) * 1e3 for k, v in loaded_results.items()}
time_results_std = {k: np.std(v.model_inference_time) * 1000 for k, v in loaded_results.items()}

In [102]:
perf_results = {}
for k, v in loaded_results.items():
  latency_list = v.model_inference_time
  latency_50 = np.percentile(latency_list, 50) * 1e3
  latency_75 = np.percentile(latency_list, 75) * 1e3
  latency_90 = np.percentile(latency_list, 90) * 1e3
  latency_95 = np.percentile(latency_list, 95) * 1e3
  latency_99 = np.percentile(latency_list, 99) * 1e3

  average_latency = np.mean(v.model_inference_time) * 1e3
  throughput = 1 * (1000 / average_latency)

  perf_results[k] = (
        average_latency,
        latency_50,
        latency_75,
        latency_90,
        latency_95,
        latency_99,
        throughput,
    )

In [103]:
import pandas as pd

index_labels = ['Average_latency (ms)', 'Latency_P50', 'Latency_P75',
                'Latency_P90', 'Latency_P95', 'Latency_P99', 'Throughput']
perf_df = pd.DataFrame(data=perf_results, index=index_labels)
perf_df

,PyTorch CPU,ONNX CPU,Quantized ONNX CPU
Average_latency (ms),2129.946476,1526.648568,983.414324
Latency_P50,2123.129511,1408.080623,1034.687645
Latency_P75,2172.325938,1839.126345,1052.753890
Latency_P90,2222.387421,1910.024937,1069.226776
Latency_P95,2272.468230,1932.687429,1079.612645
Latency_P99,2479.688826,1972.021206,1107.149125
Throughput,0.469495,0.655030,1.016865


In [104]:
results = loaded_results
results_df = pd.DataFrame(columns=['Provider', 'Inference_time'])
for k, v in results.items():
  for i in range(len(v.model_inference_time)):
    results_df.loc[len(results_df.index)] = [k, v.model_inference_time[i] * 1e3]

fig = px.box(results_df, x="Provider", y="Inference_time",
             points="all",
             labels={'Provider':'Provider', 'Inference_time':'Inference durations (ms)'})
fig.show()